In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed, specific_humidity_from_mixing_ratio
import pyart
from wrf import getvar
import haversine
from metpy.interpolate import log_interpolate_1d, interpolate_1d


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
wspd_thresh = 20 #wind speed cutoff for UAS ops, in m/s

#dt = datetime(2022,9,15,10)
dt = datetime(2022,7,19,10)
#dt = datetime(2021,6,4,10)
#New sites
# lons_newuas = [-84.55, -84.93, -84.55, -84.09, -84.15, -84.52, -84.94, -85.21, -85.29]
# lats_newuas = [39.21, 39.41, 39.48, 39.47, 38.92, 38.86, 38.95, 39.19, 38.86]

#METAR-located sites
lons_newuas = [-85.26000214,-84.77999878,-84.66999817,-84.51999664,-84.41999817,-84.40000153,-84.25,-84.23000336,-84.22000122]
lats_newuas = [39.34999847,39.25,39.04000092,39.36000061,39.09999847,39.52999878,39.45999908,39.59999847,39.08000183]
# lons_newuas = [-84.55]
# lats_newuas = [39.21]

# lons_newuas = [-84.548065]
# lats_newuas = [39.212013]

#FOGMAP Sites-IOP6
# lons_newuas = [-84.552, -84.688]
# lats_newuas = [39.083,39.060]

prof_lons = lons_newuas
prof_lats = lats_newuas

In [3]:
#Make arrays of heights and times
#For 1 km profile at a 3 m/s ascent/descent rate

#profile depth, in m
depth = 1000

#profile resolution, in m
res = 50

#uas_z = np.concatenate([np.arange(res, depth+res, res), np.arange(depth-res, 0, res*-1)])
#uas_z = np.concatenate([np.arange(res, depth+res, res)])
uas_z = np.concatenate([np.arange(res-res, depth+res, res)])

#Get time, in seconds, for this profile

#ascent rate (in m/s)
ar = 3

#uas_time = np.arange(res/ar, ((len(uas_z)*res)/ar)+(res/ar), res/ar) 
# uas_time = np.arange(0, ((len(uas_z)*res)/ar), res/ar) 
uas_time = np.arange(0, ((len(uas_z)*res)/ar), res/ar) 

print(uas_z)
print(uas_time)

print(uas_z.shape)
print(uas_time.shape)

#Replicate this profile hourly
#Interval between flights (in seconds)
fl_int = 3600
#Number of flights
fl_num = 10
#Start day
#st_day = 154024
st_day = 153966
#st_day = 153556

#first_time (in seconds)
st_sec = 3*fl_int
end_sec = (3+fl_num)*fl_int
#Get a range of times

time_range = np.arange((st_sec/86400), (end_sec/86400), fl_int/86400) + st_day

#Now get the times assuming launches at each interval

uas_times2=[]
uas_z2 = []
for time_l in time_range:
    time_prof = time_l + (uas_time/86400)
    uas_times2.append(time_prof)
    uas_z2.append(uas_z)
uas_times2 = np.concatenate(uas_times2)
uas_z2=np.concatenate(uas_z2)
print(uas_times2)

[   0   50  100  150  200  250  300  350  400  450  500  550  600  650
  700  750  800  850  900  950 1000]
[  0.          16.66666667  33.33333333  50.          66.66666667
  83.33333333 100.         116.66666667 133.33333333 150.
 166.66666667 183.33333333 200.         216.66666667 233.33333333
 250.         266.66666667 283.33333333 300.         316.66666667
 333.33333333]
(21,)
(21,)
[153966.125      153966.1251929  153966.1253858  153966.1255787
 153966.1257716  153966.12596451 153966.12615741 153966.12635031
 153966.12654321 153966.12673611 153966.12692901 153966.12712191
 153966.12731481 153966.12750772 153966.12770062 153966.12789352
 153966.12808642 153966.12827932 153966.12847222 153966.12866512
 153966.12885802 153966.16666667 153966.16685957 153966.16705247
 153966.16724537 153966.16743827 153966.16763117 153966.16782407
 153966.16801698 153966.16820988 153966.16840278 153966.16859568
 153966.16878858 153966.16898148 153966.16917438 153966.16936728
 153966.16956019 153966.1

In [4]:
#Obs types for the UAS obs. Change once we have this implemented in DART
otype_T = 123
#otype_q = 124
otype_q = 67
otype_u = 121
otype_v = 122

#Errors for UAS obs
oerr_T = 0.25
oerr_q = 0.25
#oerr_q = 0.0000005
#oerr_q = 0.0
oerr_u = 1.0
oerr_v = 1.0

otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []
pres_s = []

minute_range = np.arange(180,605,5)
#minute_range = np.arange(180,190,5)

#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg')).to('K')
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    T_z = np.asarray(getvar(wrfout, "tk"))
    p_z = np.asarray(getvar(wrfout, "pres"))
    q_zi = np.asarray(wrfout.variables['QVAPOR'][:])
    q_z = specific_humidity_from_mixing_ratio(q_zi)
    u_z, v_z = getvar(wrfout, 'uvmet')
    u_z = np.asarray(u_z)
    v_z = np.asarray(v_z)
    z_z = np.asarray(getvar(wrfout, "height_agl"))
    #z_z = np.asarray(getvar(wrfout, "height"))
    z_zm = np.asarray(getvar(wrfout, "height"))
    td_z = dewpoint_from_specific_humidity(p_z*units('Pa'), T_z*units('K'), q_z*units('kg/kg')).to('K')
    
    #Convert WRF file time into same units as the obs_seq time
    dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
    time_diff = np.abs(dt_tot - uas_times2)
    
    #Get obs within +- 2.5 minutes of each WRF file
    time_T3 = uas_times2[time_diff<(150/86400)]
    #loc_T3 = loc_T2[time_diff<(150/86400), :]
    #qc_T3 = qc_T2[time_diff<(150/86400)]
    elev_T3 = uas_z2[time_diff<(150/86400)]
    if len(time_T3)==0:
            print('NO OBS IN WINDOW')
            continue
    #Get the correct lat/lon for an example point
    for st1 in range(len(prof_lats)):
        latp=prof_lats[st1]
        lonp=prof_lons[st1]
        #esfcp = prof_esfc[st1]
        #Get location for each ob in model land
        lon1d = np.ndarray.flatten(lon[0,:,:])
        lat1d = np.ndarray.flatten(lat[0,:,:])
        station = []
        points = []
        for i in range(len(lon1d)):
            points.append((lat1d[i],lon1d[i]))
            station.append((latp,lonp))
        dist = haversine.haversine_vector(station,points)
        dist2=dist.reshape(lon.shape[1],lon.shape[2])
        print(lon[0,:,:][np.where(dist2==np.min(dist2))])
        print(lat[0,:,:][np.where(dist2==np.min(dist2))])
        print(np.where(dist2==np.min(dist2)))
        st_xind = np.where(dist2==np.min(dist2))[0][0]
        st_yind = np.where(dist2==np.min(dist2))[1][0]

        #Actually get those interpolated obs
        # z_point = z_z[:,st_xind,st_yind]
        # t_point = T_z[:,st_xind,st_yind]
        # q_point = td_z[0,:,st_xind,st_yind].magnitude
        # u_point = u_z[:,st_xind,st_yind]
        # v_point = v_z[:,st_xind,st_yind]
        # z_point = np.concatenate([[0], z_z[:,st_xind,st_yind]])
        # t_point = np.concatenate([[T2[0,st_xind,st_yind].magnitude], T_z[:,st_xind,st_yind]])
        # q_point = np.concatenate([[Td2[st_xind,st_yind].magnitude], td_z[0,:,st_xind,st_yind].magnitude])
        # #q_point = np.concatenate([[Q2[0,st_xind,st_yind]], q_z[0,:,st_xind,st_yind]])
        # u_point = np.concatenate([[U10[0,st_xind,st_yind]], u_z[:,st_xind,st_yind]])
        # v_point = np.concatenate([[V10[0,st_xind,st_yind]], v_z[:,st_xind,st_yind]])

        #Experiment to force it to get the nearest value for points below the lowest model level
        z_point = np.concatenate([[0], z_z[:,st_xind,st_yind]])
        t_point = np.concatenate([[T_z[0,st_xind,st_yind]], T_z[:,st_xind,st_yind]])
        q_point = np.concatenate([[td_z[0,0,st_xind,st_yind].magnitude], td_z[0,:,st_xind,st_yind].magnitude])
        #q_point = np.concatenate([[Q2[0,st_xind,st_yind]], q_z[0,:,st_xind,st_yind]])
        u_point = np.concatenate([[u_z[0,st_xind,st_yind]], u_z[:,st_xind,st_yind]])
        v_point = np.concatenate([[v_z[0,st_xind,st_yind]], v_z[:,st_xind,st_yind]])
        zm_point = z_zm[:,st_xind,st_yind]
        esfc = zm_point[0]-z_point[0]
        #p_point = np.concatenate([[P2[0,st_xind, st_yind].magnitude], p_z[:,st_xind,st_yind]/100])
        p_point = np.concatenate([[p_z[0,st_xind,st_yind]/100], p_z[:,st_xind,st_yind]/100])
        #p_point = p_z[:,st_xind,st_yind]/100
        
        m = 0
        for point in elev_T3:
            
            T2_a = interpolate_1d(point, z_point, t_point)
            q2_a = interpolate_1d(point, z_point, q_point)
            u2_a = interpolate_1d(point, z_point, u_point)
            v2_a = interpolate_1d(point, z_point, v_point)
            p2_a = interpolate_1d(point, z_point, p_point)

            wspd_a = np.sqrt(u2_a**2 + v2_a**2)
            if wspd_a > wspd_thresh:
                print('wind is too strong, skipping')
            else:
            
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_T))
                if np.abs(error/4) > (np.sqrt(oerr_T)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_T)*1.0)
                T2_b = T2_a + error/4
                #uas_T.append(T2_b)
                print(T2_b)
                obs_s.append(T2_b)
                otype_s.append(otype_T)
                time_s.append(time_T3[m])
                elev_s.append(point+esfc)
                error_s.append(oerr_T)
                lon_s.append(lonp)
                lat_s.append(latp)
                pres_s.append(p2_a)
                #Q section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_q))
                if np.abs(error/4) > (np.sqrt(oerr_q)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_q)*1.0)
                q2_b = q2_a + error/4
                #q2_b = q2_a
                #uas_q.append(q2_b)
                obs_s.append(q2_b)
                otype_s.append(otype_q)
                time_s.append(time_T3[m])
                elev_s.append(point+esfc)
                error_s.append(oerr_q*4)
                lon_s.append(lonp)
                lat_s.append(latp)
                pres_s.append(p2_a)
                
                # #U section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_u))
                if np.abs(error/4) > (np.sqrt(oerr_u)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_u)*1.0)
                u2_b = u2_a + error/4
                #uas_u.append(u2_b)
                obs_s.append(u2_b)
                otype_s.append(otype_u)
                time_s.append(time_T3[m])
                elev_s.append(point+esfc)
                error_s.append(oerr_u)
                lon_s.append(lonp)
                lat_s.append(latp)
                pres_s.append(p2_a)
                
                # #V section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_v))
                if np.abs(error/4) > (np.sqrt(oerr_v)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_v)*1.0)
                v2_b = v2_a + error/4
                #uas_v.append(v2_b)
                obs_s.append(v2_b)
                otype_s.append(otype_v)
                time_s.append(time_T3[m])
                elev_s.append(point+esfc)
                error_s.append(oerr_v)
                lon_s.append(lonp)
                lat_s.append(latp)
                pres_s.append(p2_a)
            
            m=m+1
    

2022-07-19 03:00:00
[-85.260086]
[39.350243]
(array([679]), array([133]))
[296.67961485]
[296.54079117]
[296.39275876]
[296.10020817]
[296.44360186]
[296.39645193]
[296.48685135]
[296.28700191]
[296.20375515]
[-84.77993]
[39.24957]
(array([579]), array([505]))
[297.81752544]
[298.51020206]
[298.66163189]
[298.69597897]
[298.76440921]
[298.38017174]
[297.97585626]
[297.49150327]
[297.42324614]
[-84.67051]
[39.04025]
(array([370]), array([591]))
[298.20906367]
[298.13224245]
[297.85590149]
[297.49422354]
[297.46470176]
[297.05781812]
[296.54592169]
[296.39751919]
[296.28577655]
[-84.51945]
[39.360283]
(array([691]), array([706]))
[297.60152267]
[297.86910685]
[298.02725489]
[297.85639516]
[297.92511241]
[297.87040566]
[297.9798608]
[297.64718809]
[297.28167644]
[-84.42023]
[39.099823]
(array([431]), array([785]))
[298.25145723]
[298.21632183]
[298.03397982]
[298.16830545]
[297.79970633]
[297.56280021]
[297.44555044]
[296.95284199]
[296.52812137]
[-84.39989]
[39.530434]
(array([862]), arr

/glade/work/mawilson/conda-envs/airplane/lib/python3.11/site-packages/metpy/calc/thermo.py:1403: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)


NO OBS IN WINDOW
2022-07-19 03:35:00


/glade/work/mawilson/conda-envs/airplane/lib/python3.11/site-packages/metpy/calc/thermo.py:1403: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)


NO OBS IN WINDOW
2022-07-19 03:40:00


...

NO OBS IN WINDOW
2022-07-19 09:25:00


/glade/work/mawilson/conda-envs/airplane/lib/python3.11/site-packages/metpy/calc/thermo.py:1403: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)


NO OBS IN WINDOW
2022-07-19 09:30:00


/glade/work/mawilson/conda-envs/airplane/lib/python3.11/site-packages/metpy/calc/thermo.py:1403: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)


NO OBS IN WINDOW
2022-07-19 09:35:00
NO OBS IN WINDOW
2022-07-19 09:40:00
NO OBS IN WINDOW
2022-07-19 09:45:00
NO OBS IN WINDOW
2022-07-19 09:50:00
NO OBS IN WINDOW
2022-07-19 09:55:00
NO OBS IN WINDOW
2022-07-19 10:00:00
[-85.260086]
[39.350243]
(array([679]), array([133]))
[293.31801118]
[294.9769908]
[295.76074378]
[295.53053962]
[295.52944367]
[295.26289218]
[295.2111778]
[294.67594575]
[294.34664263]
[-84.77993]
[39.24957]
(array([579]), array([505]))
[295.15967431]
[295.36345935]
[295.02108123]
[295.88070834]
[296.28082325]
[296.07689478]
[295.99651853]
[295.98922197]
[295.42592789]
[-84.67051]
[39.04025]
(array([370]), array([591]))
[295.57315956]
[295.81053941]
[295.99493942]
[295.98940015]
[295.76850128]
[295.13796291]
[295.54851991]
[295.34253079]
[295.50073204]
[-84.51945]
[39.360283]
(array([691]), array([706]))
[293.12543426]
[294.66992098]
[297.03906005]
[297.69593042]
[297.35577146]
[296.6576651]
[296.24884249]
[295.96722583]
[296.0546863]
[-84.42023]
[39.099823]
(array(

In [5]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
day_DART = 153966
#day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [6]:
# for ob in obs_s:
#     print(ob[0])

In [7]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
# print(seconds_DART)
# print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]
pres_s1 = np.asarray(pres_s)[inds_time]

In [8]:
print(pres_s1)

[[980.3984375 ]
 [980.3984375 ]
 [980.3984375 ]
 ...
 [935.42048705]
 [935.42048705]
 [935.42048705]]


In [9]:
for bigfoot in [1,2]:
    print(bigfoot)
    #Write the simulated obs out to an obs_seq file
    #filename = 'SIM_UAS_IOP6_T_grid'
    #filename = 'SIM_UAS_IOP6_FOGMAP'
    filename = 'SIM_UAS_IOP4_NOSFC9'
    
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(4))
    fi.write("    %d          %s   \n" %(123, 'UAS_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(67, 'UAS_DEWPOINT'))
    #fi.write("    %d          %s   \n" %(124, 'UAS_SPECIFIC_HUMIDITY'))
    fi.write("    %d          %s   \n" %(121, 'UAS_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(122, 'UAS_V_WIND_COMPONENT'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        # fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
        #                    (lon_DART1[q], lat_DART1[q], pres_s1[q]*100, 2))
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], 3))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


/glade/derecho/scratch/mawilson/tmp/ipykernel_17519/4067711171.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fi.write("   %20.14f\n" % obs_s1[q] )


In [10]:
print((st_sec/86400))

0.125


In [11]:
print(fl_num*(st_sec/86400))

1.25


In [12]:
print(np.asarray(obs_s1))
print(np.asarray(elev_s1))

[[297.09642942]
 [294.39176023]
 [ -0.91303944]
 ...
 [292.0095825 ]
 [  5.30493869]
 [ -2.27962144]]
[271.50665283 271.50665283 271.50665283 ... 705.34490967 705.34490967
 705.34490967]
